In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from math import factorial
from scipy.ndimage import rotate

# Создаем директорию для хранения датасета
dataset_dir = 'DataSet_single' 
if not os.path.exists(dataset_dir):
    os.makedirs(dataset_dir)

# Вспомогательная функция для вычисления радиальной части полиномов Цернике
def R(n, m, r):
    """Функция для расчета радиальной части полинома Цернике"""
    if (n - m) % 2 != 0:
        return np.zeros_like(r)  # Если разность нечетная, результат 0

    radial_sum = np.zeros_like(r)
    for k in range((n - m) // 2 + 1):
        c = (-1) ** k * factorial(n - k) / (factorial(k) * factorial((n + m) // 2 - k) * factorial((n - m) // 2 - k))
        radial_sum += c * r ** (n - 2 * k)
    return radial_sum

import numpy as np
import matplotlib.pyplot as plt

# Бяка с дробями в экспоненте (Гаус-Л) m - int
def multi_channel_filter(n,m,a, Sg, C, grid_size=100):
    y, x = np.linspace(-1, 1, grid_size), np.linspace(-1, 1, grid_size)
    #alph = [10, 15, 20, 25]
    Ln = [0, 1, 5, 7]

    X, Y = np.meshgrid(x, y)
    R = np.sqrt(X ** 2 + Y ** 2)


    Gs = np.exp(-(R ** 2) / (Sg ** 2)) 
              


    Fl = ((X + 1j * Y) ** (Ln[0]) * (Ln[1]) * np.exp(1j * C * X + 1j * C * Y) + 
          (X + 1j * Y) ** (Ln[1]) * (Ln[1]) *np.exp(1j * (-C) * X + 1j * C * Y) + 
          (X + 1j * Y) ** (Ln[2]) * (Ln[2]) *np.exp(1j * C * X + 1j * (-C) * Y) + 
          (X + 1j * Y) ** (Ln[3]) * (Ln[3]) *np.exp(1j * (-C) * X + 1j * (-C) * Y))
    
    rho = np.sqrt(X**2 + Y**2)
    theta = np.arctan2(Y, X)
    
    # Ограничиваем область единичным кругом
    mask = rho <= 1
    rho[~mask] = 0
    theta[~mask] = 0
    
    Z = zernike_polynomial(n, m, rho, theta)
    Z1 = np.exp(1j * a * Z)
    Fl = Fl * Z1 * Gs

    Flf = np.fft.fftshift(np.fft.fft2(Fl, s=[1000, 1000]))

    # Создание одного графика (не сетки)
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))

    # Исправлено: ax - это одиночный объект Axes, а не массив
    img3 = ax.imshow(np.abs(Flf), extent=(-1, 1, -1, 1), cmap='RdBu')
    ax.set_title(f"MultiChannel Filter (C={C})")
    plt.colorbar(img3, ax=ax)
    plt.show()
    
    return Flf

# Основная функция для вычисления полиномов Цернике
def zernike_polynomial(n, m, rho, theta):
    """Вычисление полинома Цернике"""
    if m >= 0:
        return R(n, m, rho) * np.cos(m * theta)
    else:
        return R(n, -m, rho) * np.sin(-m * theta)

def noise(img, p):
  rows, cols = img.shape
  noise = (np.random.rand(rows, cols) < p) * np.random.rand(rows, cols) * (np.max(img)/4)
  return img + noise 

# Функция для обрезки изображения до центральной части 128x128
def crop_center(img, crop_size=128):
    h, w = img.shape
    startx = w // 2 - (crop_size // 2)
    starty = h // 2 - (crop_size // 2)    
    return img[starty:starty + crop_size, startx:startx + crop_size]

# Функция для создания аугментаций изображений
def augment_image(img, count):
    augmented_images = []
    for i in range(count):
        # Применение различных аугментаций
        rotated_img = rotate(img, angle=np.random.randint(0, 360), reshape=False)
        noise_img1 = noise(rotated_img, 0.04)
        noise_img2 = noise(rotated_img, 0.08)
        # noise_img = rotated_img + np.random.normal(0, np.random.uniform(0, 1), rotated_img.shape)        
        augmented_images.extend([rotated_img, noise_img1, noise_img2])
    return augmented_images

# Функция для генерации и сохранения изображений
def generate_images(n, m, a_range, class_name, grid_size=100, num_augmentations=30):
    y, x = np.linspace(-1, 1, grid_size), np.linspace(-1, 1, grid_size)
    X, Y = np.meshgrid(x, y)
    rho = np.sqrt(X**2 + Y**2)
    theta = np.arctan2(Y, X)
    
    # Ограничиваем область единичным кругом
    mask = rho <= 1
    rho[~mask] = 0
    theta[~mask] = 0
    
    class_dir = os.path.join(dataset_dir, class_name)
    if not os.path.exists(class_dir):
        os.makedirs(class_dir)
    
    image_counter = 0
    b = 25
    for a in a_range:
        # Генерация полинома Цернике
        Z = zernike_polynomial(n, m, rho, theta)
        Z1 = np.exp(1j * a * Z)
        #Z1a = np.cos((a * Z + b * np.sqrt(X ** 2 + Y ** 2)))
        Z1a = np.exp( 1j * (a * Z + b * np.sqrt(X ** 2 + Y ** 2)))

        Z1af = np.fft.fftshift(np.fft.fft2(Z1a, s=[512, 512]))
        
        # Фаза полинома Цернике
        aberration_img = abs(Z1af)
        
        # Сохранение изображений
        
        plt.imsave(os.path.join(class_dir, f'zernike_aberration_n{n}_m{m}_a{a:.2f}_b{b:.2f}.png'), crop_center(aberration_img), cmap='gray')
        
        # # Применение аугментаций и сохранение
        # aberration_augmentations = augment_image(aberration_img, num_augmentations)
        
        # for i, aug_img in enumerate(aberration_augmentations):
        #     plt.imsave(os.path.join(class_dir, f'zernike_aberration_n{n}_m{m}_a{a:.2f}_b{b:.2f}_aug{i}.png'), crop_center(aug_img), cmap='gray')
                
        image_counter += 1
        if image_counter % 100 == 0:
            print(f'{image_counter} изображений сгенерировано')

# Параметры для генерации
aberration_classes = [
    (2, 2, 'class1'), 
    (3, 1, 'class2'), 
    (3, 3, 'class3'), 
    (4, 2, 'class4'), 
    (4, 4, 'class5'),
    (5, 1, 'class6'), 
    (5, 3, 'class7'),
    (5, 5, 'class8'),
]
a_range_large = np.linspace(1, 10, 3)  # Амплитуда a

# Генерация изображений для каждого класса
for n, m, class_name in aberration_classes:
    print("class: ", class_name)
    generate_images(n, m, a_range_large, class_name)


class:  class1
class:  class2
class:  class3
class:  class4
class:  class5
class:  class6
class:  class7
class:  class8


: 

In [17]:
import numpy as np
import matplotlib.pyplot as plt
import os

# Define the class names and their corresponding aberration types
aberration_classes = [
    (2, 2, 'class1', 'Astigmatism (n=2, m=2)'),
    (3, 1, 'class2', 'Coma (n=3, m=1)'),
    (3, 3, 'class3', 'Trefoil (n=3, m=3)'),
    (4, 2, 'class4', 'Secondary Astigmatism (n=4, m=2)'),
    (4, 4, 'class5', 'Quatrefoil (n=4, m=4)'),
    (5, 1, 'class6', 'Secondary Coma (n=5, m=1)'),
    (5, 3, 'class7', 'Secondary Trefoil (n=5, m=3)'),
    (5, 5, 'class8', 'Secondary Quatrefoil (n=5, m=5)')
]

# Dataset parameters
images_per_class = 11250  # Specified number of images per class
num_classes = len(aberration_classes)  # 8 classes
total_images = images_per_class * num_classes  # 11,250 * 8 = 90,000 images

# Data for histogram
class_names = [name for _, _, _, name in aberration_classes]
image_counts = [images_per_class] * num_classes  # Equal distribution of 11,250 images per class

# Create histogram
plt.figure(figsize=(10, 6))
plt.bar(class_names, image_counts, color='gray', edgecolor='black')  # Grayscale bars
plt.xlabel('Aberration Class', color='black')
plt.ylabel('Number of Images', color='black')
plt.title('Dataset Distribution Across Zernike Aberration Classes', color='black')
plt.xticks(rotation=45, ha='right', color='black')
plt.yticks(color='black')
plt.grid(axis='y', linestyle='--', color='black', alpha=0.5)  # Black-and-white grid
plt.ylim(0, images_per_class * 1.1)  # Extend y-axis to 110% of max count

# Add text annotations for image counts with reduced offset
for i, count in enumerate(image_counts):
    plt.text(i, count + 200, f'{count:,}', ha='center', va='bottom', fontsize=10, color='black')

# Tight layout with padding to prevent overlap with title
plt.tight_layout(pad=2.0)

# Save the histogram
output_dir = 'figures'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
plt.savefig(os.path.join(output_dir, 'class_distribution_histogram_11250_bw.png'), dpi=300, bbox_inches='tight', facecolor='white', edgecolor='black')
plt.close()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from math import factorial
from scipy.ndimage import rotate

# Вспомогательная функция для вычисления радиальной части полиномов Цернике
def R(n, m, r):
    """Функция для расчета радиальной части полинома Цернике"""
    if (n - m) % 2 != 0:
        return np.zeros_like(r)  # Если разность нечетная, результат 0

    radial_sum = np.zeros_like(r)
    for k in range((n - m) // 2 + 1):
        c = (-1) ** k * factorial(n - k) / (factorial(k) * factorial((n + m) // 2 - k) * factorial((n - m) // 2 - k))
        radial_sum += c * r ** (n - 2 * k)
    return radial_sum

# Основная функция для вычисления полиномов Цернике
def zernike_polynomial(n, m, rho, theta):
    """Вычисление полинома Цернике"""
    if m >= 0:
        return R(n, m, rho) * np.cos(m * theta)
    else:
        return R(n, -m, rho) * np.sin(-m * theta)

def noise(img, p):
  rows, cols = img.shape
  noise = (np.random.rand(rows, cols) < p) * np.random.rand(rows, cols) * (np.max(img)/4)
  return img + noise 

# Функция для обрезки изображения до центральной части 128x128
def crop_center(img, crop_size=128):
    h, w = img.shape
    startx = w // 2 - (crop_size // 2)
    starty = h // 2 - (crop_size // 2)    
    return img[starty:starty + crop_size, startx:startx + crop_size]

# Функция для создания аугментаций изображений
def augment_image(img, count):
    augmented_images = []
    for i in range(count):
        # Применение различных аугментаций
        rotated_img = rotate(img, angle=np.random.randint(0, 360), reshape=False)
        noise_img1 = noise(rotated_img, 0.04)
        noise_img2 = noise(rotated_img, 0.08)
        # noise_img = rotated_img + np.random.normal(0, np.random.uniform(0, 1), rotated_img.shape)        
        augmented_images.extend([rotated_img, noise_img1, noise_img2])
    return augmented_images

# Функция для генерации и сохранения изображений
def generate_images(n, m, a_range, class_name, grid_size=100, num_augmentations=30):
    y, x = np.linspace(-1, 1, grid_size), np.linspace(-1, 1, grid_size)
    X, Y = np.meshgrid(x, y)
    rho = np.sqrt(X**2 + Y**2)
    theta = np.arctan2(Y, X)
    
    # Ограничиваем область единичным кругом
    mask = rho <= 1
    rho[~mask] = 0
    theta[~mask] = 0
    
    class_dir = os.path.join(dataset_dir, class_name)
    if not os.path.exists(class_dir):
        os.makedirs(class_dir)
    
    image_counter = 0
    b = 25
    for a in a_range:
        # Генерация полинома Цернике
        Z = zernike_polynomial(n, m, rho, theta)
        Z1 = np.exp(1j * a * Z)
        #Z1a = np.cos((a * Z + b * np.sqrt(X ** 2 + Y ** 2)))
        Z1a = np.exp( 1j * (a * Z + b * np.sqrt(X ** 2 + Y ** 2)))

        Z1af = np.fft.fftshift(np.fft.fft2(Z1a, s=[512, 512]))
        
        # Фаза полинома Цернике
        aberration_img = abs(Z1af)
        
        # Сохранение изображений
        
        plt.imsave(os.path.join(class_dir, f'zernike_aberration_n{n}_m{m}_a{a:.2f}_b{b:.2f}.png'), crop_center(aberration_img), cmap='gray')
        
        # Применение аугментаций и сохранение
        aberration_augmentations = augment_image(aberration_img, num_augmentations)
        
        for i, aug_img in enumerate(aberration_augmentations):
            plt.imsave(os.path.join(class_dir, f'zernike_aberration_n{n}_m{m}_a{a:.2f}_b{b:.2f}_aug{i}.png'), crop_center(aug_img), cmap='gray')
                
        image_counter += 1
        if image_counter % 100 == 0:
            print(f'{image_counter} изображений сгенерировано')

# Параметры для генерации
aberration_classes = [
    (2, 2, 'class1'), 
    (3, 1, 'class2'), 
    (3, 3, 'class3'), 
    (4, 2, 'class4'), 
    (4, 4, 'class5'),
    (5, 1, 'class6'), 
    (5, 3, 'class7'),
    (5, 5, 'class8'),
]
a_range_large = np.linspace(0.1, 2, 20)  # Амплитуда a

# Генерация изображений для каждого класса
for n, m, class_name in aberration_classes:
    print("class: ", class_name)
    generate_images(n, m, a_range_large, class_name)
